# Testing Natural Language Understanding of a Conversational In-Car Chatbot 🚗

# Goal

You are going to test the Natural Language Understanding (NLU) component of a task-oriented chatbot with STELLAR.
Specifically, the goal is to generate natural language inputs that expose failures in the chatbot's intent recognition capabilities to control the climate control.

## What is a test input?

A test input is a natural language utterance.

## What is a failure? 

A test is considered a failure, when the expected intent of an utterance, is not detected by the chatbot.

Note: To simplify the overall task, there is no further parametrization of intents/slot filling performed.

## Example Pass

- **Utterance:** "Can you start the fan?"
- **Detected:** `INTENT_ActivateFan`

## Example Fail

- **Utterance:** "I am freezing one's buns off!"
- **Detected:** `INTENT_Unknown`

## Task

Your task is to improve the notebooks and provide the following components to apply STELLAR to test NLU:

1. Utterance Generator (Prompts)
2. Fitness Function

## Ranking

The results will be ranked based on the following criteria:

- Fail Rate - number of failures divided by number of generated tests (the higher the better)
- Diversity - number of different intents failed (the higher the better)
- Number Tokens - number of input and output tokens spent for test generation (the lower the better)


# Getting Started

Let's try out some utterancies to get familiar with the system and goal.

Run the following command to start the chatbot. A browser window should pop up. You should be able to find the deployment otherwise here. You can send the chatbot utterance, and it will respond with intents. 

All possible intents that can be detected by assistant are provided here: [nlu_bot/intents.py](nlu_bot/intents.py).

You may need to restart the kernel to exit the dashboard.

In [ ]:
from pathlib import Path; import subprocess, sys
app = next((p for p in (Path.cwd() / "nlu_bot" / "chatbot_app.py", Path.cwd() / "jupyter" / "nlu_bot" / "chatbot_app.py", Path.cwd().parent / "jupyter" / "nlu_bot" / "chatbot_app.py") if p.exists()), None)
subprocess.run([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app),
    "--server.headless",
    "true",
], check=True)

# Automated Testing

Manually testing this chatbot is challenging. Let us apply STELLAR to test the bot.

Run the following code to automatically generate the test cases.

The generation should take 1 minute.

- **Linux / macOS:** run the first cell below (uses `bash`).
- **Windows:** run the second cell below (pure Python, no `bash` required).


In [ ]:
# Linux / macOS
from pathlib import Path
script = next((p for p in (Path.cwd() / "run_test_script.sh", Path.cwd() / "jupyter" / "run_test_script.sh", Path.cwd().parent / "jupyter" / "run_test_script.sh", Path.cwd().parent.parent / "jupyter" / "run_test_script.sh") if p.exists()), None)
!bash {script}


In [ ]:
# Windows
from pathlib import Path
import os
import subprocess
import sys

script = next((p for p in (Path.cwd() / "run_test_script.sh", Path.cwd() / "jupyter" / "run_test_script.sh", Path.cwd().parent / "jupyter" / "run_test_script.sh", Path.cwd().parent.parent / "jupyter" / "run_test_script.sh") if p.exists()), None)
repo_root = script.parent.parent

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(repo_root), env.get("PYTHONPATH", "")]))

# Stream stdout+stderr live so the real error from jupyter.run_test is visible in the cell.
process = subprocess.Popen(
    [sys.executable, "-m", "jupyter.run_test"],
    cwd=str(repo_root),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise SystemExit(f"jupyter.run_test failed with exit code {return_code} (see output above).")


# STELLAR Dashboard

Let us visualize the results in a dashboard http://localhost:8501/. 
You may need to restart the kernel to exit the dashboard.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

dashboard = next((p for p in (Path.cwd() / "dashboard.py", Path.cwd().parent / "dashboard.py", Path.cwd().parent.parent / "dashboard.py") if p.exists()), None)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Stream stdout+stderr live so the real Streamlit error is visible in the cell.
process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(dashboard),
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise SystemExit(f"streamlit run failed with exit code {return_code} (see output above).")


# Your turn

Now it is your turn: improve the implementation of the test generator to find more failures.

Adjust the different notebooks below. You can just adjust some of the components if you are not sure
about your implementation. Test your code at the end in step 4. 


# 1. Utterance Generator

Implement an utterance generator which creates user inputs based on provided feature values. 

Example of feature dictionary

```
{'intent': 'INTENT_ActivateSeatCooling', 'word_perturbation': None, 'slang': 'neutral', 'implicitness': 'not implicit', 'politeness': 'unpolite', 'anthropomorphism': 'directive'}
```


In [ ]:
from pathlib import Path
import sys
from typing import Any, Dict

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents[:2]) if (p / "examples").exists() and (p / "jupyter").exists()), None)

if project_root is None:
    raise FileNotFoundError("Could not locate the STELLAR project root")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from examples.navi.models import StyleDescription
from jupyter.misc.models import NLUContentInput
from jupyter.misc.nlu_utterance_generator import NLUUtteranceGenerator
from jupyter.nlu_bot.intents import INTENTS, INTENT_SET

PROMPT_GENERATOR = """ You are an intelligent user request generator to test an in car assistant."""
FALLBACK_INTENT = "INTENT_Unknown"

class CustomNLUUtteranceGenerator(NLUUtteranceGenerator):
    def build_style_prompt(
        self,
        feature_values: Dict[str, Any],
    ) -> str:
        num_words = "num_words"
        result = ""
        if num_words in feature_values:
            result += (
                f"The utterance must contain exactly "
                f"{feature_values[num_words]} words\n"
            )

        style_description = StyleDescription.model_validate(feature_values)

        result += style_description.model_dump_json(
            exclude_none=True,
            indent=2,
        )

        return "The linguistic and style features are:\n" + result

    def build_content_prompt(
        self,
        content_input: NLUContentInput,
    ) -> str:
        intent_name = (
            content_input.intent
            if content_input.intent in INTENT_SET
            else FALLBACK_INTENT
        )
        allowed_intents = "\n".join(
            f"- {intent}" for intent in INTENTS
        )

        return f"""
INTENT:
{intent_name}

ALLOWED_INTENTS:
{allowed_intents}

RULE:
Return one natural utterance expressing the intent.
Use only the selected intent above as semantic target.

The utterance should be shorter than 15 words.
Prefer shorter utterances.

Do not include variables or placeholders.

The language must be: ENGLISH
"""

    def build_perturbation_prompt(
        self,
        feature_values: Dict[str, Any],
    ) -> str:
        result = ""
        if "word_perturbation" in feature_values:
            if feature_values["word_perturbation"] == "introduce_fillers_llm_combined":
                result = """
Apply also at the very end the following perturbation:

Insert 1-2 natural filler words into the text
to make it sound more conversational and natural.

Return ONLY the modified text
with fillers inserted.

Use common English filler words like:
\"um\", \"uh\", \"well\", \"like\", \"you know\",
\"actually\", \"I mean\", \"sort of\", \"kind of\",
or others if you think they are relevant.

IMPORTANT:
- Insert fillers at natural pause points
(not in the middle of phrases)
- Keep the original meaning and flow
- Use fillers that fit the conversational tone
- Don't overuse fillers
- 1-2 insertions maximum
- Maintain original punctuation and capitalization
"""
            elif feature_values["word_perturbation"] == "introduce_homophones_llm_combined":
                result = """
At the very end, apply the following perturbation:

Replace at least one and at most two words
in the text with valid English homophones
(words that sound the same
but are spelled differently).

Return ONLY the modified text
with the substitutions applied.

Requirements:
- Use only real, valid homophones
- Preserve original capitalization
and punctuation
- If no suitable homophones are available,
return the text unchanged
"""

        return result

    def build_utterance_prompt(
        self,
        style_prompt_text: str,
        content_prompt_text: str,
        perturbation_prompt_text: str,
    ) -> str:
        return f"""You are an intelligent, human-like utterance generator and know how people talk. Your task is to generate natural utterances considering the style attributes, content, and perturbation features defined below.

Style:
{style_prompt_text}

Content:
{content_prompt_text}

Perturbations:
{perturbation_prompt_text}

Guidelines:
- Styles:
    Slang (Slangy):
    Use German slang or colloquial expressions.

    Examples:
    - \"Where can I grab some food?\"
    - \"Take me to the nearest spot.\"
    - \"I need a place to chill.\"


    Implicit (Implicit):
    Ask indirectly without naming the venue explicitly.

    Examples:
    - \"Where can I still get something warm?\"
    - \"My car is making strange noises.\"
    - \"I need somewhere to sleep.\"


    Politeness (Rude):
    Sound unfriendly, impatient, or insulting.

    Examples:
    - \"Hurry up already.\"
    - \"Where is that damn place?\"
    - \"Start driving now.\"


    Anthropomorphism (very directive):
    Make the utterance very short and directive.

    Examples:
    - \"To the station.\"
    - \"Nearest gas station.\"
    - \"Hospital now.\"

Return only one utterance.
"""

    def build_system_prompt(self) -> str:
        return PROMPT_GENERATOR

# 2. Fitness Function

Implement fitness functions which guides.

Simulation output is an object containing
 - `.utterance` user utterance (`utterance.question`)
 - `.raw_output` in form `{'llm_output': '{intent_confidences:[{intent_index:0,confidence:0.01},{intent_index:1,confidence:0.01},{intent_index:2,confidence:0.01},{intent_index:3,confidence:0.9},...}`
 - `.top_intents` in form ` [{'intent': 'INTENT_ActivateSeatCooling', 'confidence': 0.9}, {'intent': 'INTENT_ActivateAirConditioning', 'confidence': 0.01}, {'intent': 'INTENT_ActivateClimateSync', 'confidence': 0.01}]`

In [ ]:
from typing import Tuple

from opensbt.evaluation.fitness import Fitness
from llm.model.qa_simout import QASimulationOutput
from llm.utils.embeddings_openai import get_similarity
from jupyter.nlu_bot.intents import INTENT_SET

FALLBACK_INTENT = "INTENT_Unknown"

"""
The fitness function defines which objectives to optimize to find failures.
You can add an objective by using e.g. the confidence in the prediction to direct the search.
Dont forget: Adjust the fitness name, the min_max direction, if you add another objective.
"""

class CustomIntentMatchFitness(Fitness):

    def __init__(self, llm_type=None):
        self.llm_type = llm_type
        super().__init__()

    @property
    def min_or_max(self):
        """Return 'min' if the fitness should be minimized, 'max' if it should be maximized.
           For each objective assign on direction.
        """
        return ("min",)

    @property
    def name(self):
        """Name of the objective. You can add multiple objectives by returning a tuple of names."""
        return ("intent_match_score",)

    # ------------------------------------------------------------
    # EVALUATION
    # ------------------------------------------------------------
    def _evaluate(
        self,
        simout: QASimulationOutput,
    ) -> Tuple[float, dict]:

        """ This fitness function evaluates right now the intent match and returns the embeddings distance
            between predicted and expected intent.
        """
        print("Evaluating fitness for utterance:", simout)
        raw = simout.utterance.raw_output
        content_input = simout.utterance.content_input

        if raw is None or content_input is None:
            return 1.0, {}

        # the predicted intent
        pred_intent = raw.get("intent", FALLBACK_INTENT)
        
        # the highest prediction certainty (not used right now)
        certainty = raw.get("certainty")

        # the expected intent
        true_intent = getattr(content_input, "intent", None)

        # all intent confidences (not used right now)
        # "intent_confidences": [
        #     {
        #         "intent": "INTENT_ActivateAirConditioning",
        #         "confidence": 0.05
        #     },
        #     {
        #         "intent": "INTENT_ActivateClimateSync",
        #         "confidence": 0.05
        #     },
        # ]
        all_intent_confidences =  raw.get("intent_confidences")

        if pred_intent not in INTENT_SET:
            pred_intent = FALLBACK_INTENT

        if pred_intent != true_intent:
            sim = get_similarity(pred_intent, true_intent)
            debug = {
                "certainty": certainty,
            }
            return float(sim), debug

        debug = {
            "certainty": certainty,
        }
        return 1.0, debug

    # ------------------------------------------------------------
    # PUBLIC API
    # ------------------------------------------------------------
    def eval(
        self,
        simout: QASimulationOutput,
        **kwargs,
    ) -> Tuple[float]:

        if simout is None or simout.utterance is None:
            return (1.0,)

        score, debug = self._evaluate(
            simout
        )
    
        # We add debug information for the analysis
        if simout.utterance.raw_output is not None:
            simout.utterance.raw_output[
                "_fitness_debug"
            ] = debug

        # The final score
        return (score,)

# 3. Test Your Implementation

Run the following cells to test your implementation.
There are no changes to the code required. 

When the execution is finished, you should see a final output like this:
```json
{
    "fail_rate": 0.0, 
    "num_diverse_fails": 0,
    "tokens_spent": 117746
}
```

In [ ]:
# Set the test parameters first to try different search configurations

population_size = 5           # number of tests per generation
max_time = "00:01:00"         # total execution time for the search, e.g., "00:00:10" for 10 seconds
n_generations = 20            # if max_time is set, n_generations will be ignored
seed = 1


Run your code by executing the cell below.

In [ ]:
from argparse import Namespace
import csv
import json
import os
from pathlib import Path
import sys
import warnings

import wandb
from opensbt.algorithm.nsga2_optimizer import NsgaIIOptimizer
from opensbt.algorithm.ps_rand import PureSamplingRand
from opensbt.config import LOG_FILE
from opensbt.utils.log_utils import (
    disable_pymoo_warnings,
    log,
    setup_logging,
 )

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
project_root = next((path for path in candidate_roots if (path / "llm").exists()), None)

if project_root is None:
    raise FileNotFoundError("Could not locate the STELLAR project root")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from llm.eval.critical import CriticalByFitnessThreshold, CriticalMerged
from llm.llms import LLMType
from llm.model.qa_problem import QAProblem
from llm.model.search_configuration import QASearchConfiguration, QASearchOperators
from llm.operators.utterance_crossover_discrete import UtteranceCrossoverDiscrete
from llm.operators.utterance_duplicates import UtteranceDuplicateEliminationDistance
from llm.operators.utterance_mutator_discrete import UtteranceMutationDiscrete
from llm.operators.utterance_repair import UtteranceRepairQuestionGenerator
from llm.operators.utterance_sampling_discrete import (
    UtteranceSamplingDiscrete,
    UtteranceSamplingGrid,
 )
from jupyter.nlu_bot.ipa_nlu_bot import IPA_NLU_BOT

def configure_runtime() -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    setup_logging(LOG_FILE)
    disable_pymoo_warnings()
    if wandb.run is None:
        wandb.init(mode="disabled")

def build_search_config(args):
    llm_generator = LLMType(args.generator)

    operators = QASearchOperators(
        crossover=UtteranceCrossoverDiscrete(
            llm_type=llm_generator,
            generate_question=True,
        ),
        sampling=(
            UtteranceSamplingGrid(
                llm_type=llm_generator,
                total_samples=args.population_size,
                t=4,
            )
            if args.algorithm == "random"
            else UtteranceSamplingDiscrete(
                llm_type=llm_generator,
                generate_question=True,
            )
        ),
        mutation=UtteranceMutationDiscrete(
            llm_type=llm_generator,
            generate_question=True,
        ),
        duplicate_elimination=UtteranceDuplicateEliminationDistance(),
        repair=UtteranceRepairQuestionGenerator(llm_type=llm_generator),
    )

    config = QASearchConfiguration(operators=operators)
    config.population_size = args.population_size
    config.n_generations = args.n_generations
    config.maximal_execution_time = args.max_time
    config.results_folder = args.results_folder
    return config

def build_optimizer(args, problem, config):
    if args.algorithm == "nsga2":
        return NsgaIIOptimizer(problem=problem, config=config)

    if args.algorithm == "random":
        return PureSamplingRand(problem=problem, config=config)

    raise ValueError("Unknown algorithm")

def report_execution_metrics(results_folder: str) -> dict:
    results_path = Path(results_folder)

    summary_results_path = results_path / "summary_results.csv"
    all_critical_utterances_path = results_path / "all_critical_utterances.json"
    llm_usage_summary_path = results_path / "llm_usage_summary.json"

    fail_rate = None
    with summary_results_path.open("r", encoding="utf-8", newline="") as csv_file:
        for row in csv.DictReader(csv_file):
            if row.get("Attribute") == "Ratio Critical/All scenarios (duplicate free)":
                fail_rate = float(row["Value"])
                break

    if fail_rate is None:
        raise ValueError(
            "Could not find 'Ratio Critical/All scenarios (duplicate free)' in summary_results.csv"
        )

    with all_critical_utterances_path.open("r", encoding="utf-8") as json_file:
        all_critical_utterances = json.load(json_file)

    diverse_intents = set()
    for entry in all_critical_utterances:
        utterance = entry.get("utterance", {})
        content_input = utterance.get("content_input", {})
        intent = content_input.get("intent")
        if intent:
            diverse_intents.add(intent)

    with llm_usage_summary_path.open("r", encoding="utf-8") as json_file:
        llm_usage_summary = json.load(json_file)

    tokens_spent = llm_usage_summary["total"]["tokens"]

    metrics = {
        "fail_rate": fail_rate,
        "num_diverse_fails": len(diverse_intents),
        "tokens_spent": tokens_spent,
    }

    metrics_path = results_path / "execution_metrics.json"
    with metrics_path.open("w", encoding="utf-8") as json_file:
        json.dump(metrics, json_file, indent=4)

    print(json.dumps(metrics, indent=4))
    return metrics

feature_config_path = project_root / "jupyter" / "configs" / "features_nlu.json"

args = Namespace(
    population_size=population_size,
    n_generations=n_generations,
    seed=seed,
    max_time=max_time,
    algorithm="nsga2",
    results_folder="results",
    features_config=str(feature_config_path),
    generator=LLMType.GPT_4O_MINI,
 )

configure_runtime()
log.info("Starting notebook NLU search experiment")

config = build_search_config(args)
fitness = CustomIntentMatchFitness()
critical = CriticalMerged(
    fitness_names=fitness.name,
    criticals=[
        (CriticalByFitnessThreshold(mode="<", score=1), ["intent_match_score"]),
    ],
    mode="or",
 )

problem = QAProblem(
    problem_name=(
        f"NLU_test_{args.algorithm}_p{config.population_size}"
        f"_g{config.n_generations}_t-{args.max_time}_s{args.seed}"
    ).replace(":", "-").replace("/", "-"),
    scenario_path=str(project_root),
    xl=[0],
    xu=[1],
    simulation_variables=["utterance"],
    fitness_function=fitness,
    critical_function=critical,
    simulate_function=IPA_NLU_BOT.simulate,
    seed_utterances=["play some radio"],
    context={},
    seed=args.seed,
    names_dim_utterance=["utterance"],
    feature_handler_config_path=str(feature_config_path),
    question_generator=CustomNLUUtteranceGenerator(),
 )

optimizer = build_optimizer(args, problem, config)
result = optimizer.run()
result.write_results(
    results_folder=optimizer.save_folder,
    params=optimizer.parameters,
    search_config=config,
 )

results_root = Path(optimizer.save_folder)
if results_root.is_absolute():
    try:
        results_root = results_root.relative_to(project_root)
    except ValueError:
        results_root = Path(os.path.relpath(results_root, project_root))
os.environ["STELLAR_RESULTS_ROOT"] = str(results_root)

metrics = report_execution_metrics(optimizer.save_folder)